#Ollama Configuration

In [35]:
# Installing ollama
!apt-get install -y -qq zstd > /dev/null 2>&1 || (apt-get update -qq > /dev/null 2>&1 && apt-get install -y -qq zstd > /dev/null 2>&1)
!curl -fsSL https://ollama.com/install.sh | sh 2>&1 | tail -n 2

>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [36]:
# Running ollama server
import subprocess
import time
import requests

OLLAMA_URL = "http://localhost:11434"

def server_is_up():
    try:
      return requests.get(f"{OLLAMA_URL}/api/version", timeout=2).ok
    except requests.RequestException:
      return False

if not server_is_up():
    subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(30):
        if server_is_up(): break
        time.sleep(1)

print("Server running:", server_is_up())

Server running: True


In [37]:
# Pull models
!ollama pull qwen2.5:3b
!ollama pull nomic-embed-text
!ollama list



NAME                       ID              SIZE      MODIFIED               
nomic-embed-text:latest    0a109f422b47    274 MB    Less than a second ago    
qwen2.5:3b                 357c53fb659c    1.9 GB    Less than a second ago    


#LLM Client

In [38]:
# Installing opeanai client
!pip -q install chromadb openai

In [39]:
import os, re, glob, json
from openai import OpenAI
import chromadb

OLLAMA_URL = "http://localhost:11434"
GEN_MODEL  = "qwen2.5:3b"
EMB_MODEL  = "nomic-embed-text"
CORPUS_DIR = "corpus/md"
CHROMA_DIR = "chroma"

client = OpenAI(base_url=f"{OLLAMA_URL}/v1", api_key="ollama")

#Embbedings

In [40]:
def embed(texts, model=EMB_MODEL):
    r = requests.post(f"{OLLAMA_URL}/api/embed", json={"model": model, "input": texts})
    r.raise_for_status()
    return r.json()["embeddings"]

def build_index(rows, name, persist_dir=CHROMA_DIR, batch=32, rebuild=False):
    store = chromadb.PersistentClient(path=persist_dir)

    if rebuild:
        try:
            store.delete_collection(name)
        except Exception:
            pass

    col = store.get_or_create_collection(
        name,
        metadata={"hnsw:space": "cosine", "embedding_model": EMB_MODEL},
    )

    if col.count() == len(rows):
        return col

    for i in range(0, len(rows), batch):
        part = rows[i : i + batch]
        col.upsert(
            ids=[r["id"] for r in part],
            documents=[r["text"] for r in part],
            embeddings=embed([r["text"] for r in part]),
            metadatas=[
                {"doc": r["doc"], "chunk": r["chunk"], "section": r["section"]}
                for r in part
            ],
        )
    return col

def retrieve(col, question, k=4):
    assert col.metadata.get("embedding_model") == EMB_MODEL, "Indexing and querying with different embedding models"
    res = col.query(query_embeddings=embed([question]), n_results=k, include=["documents", "metadatas", "distances"])
    return [{"text": t, "doc": m["doc"], "section": m["section"], "distance": d}
            for t, m, d in zip(res["documents"][0], res["metadatas"][0], res["distances"][0])]

#Corpus

In [41]:
os.makedirs(CORPUS_DIR, exist_ok=True)

CORPUS = {
    "datas_e_prazos.md": (
        "# Eleições Gerais 2026 · Guia de Calendário · Seção 1 · Datas e Prazos"
        " Oficiais\n\nEste documento detalha o cronograma oficial para as"
        " Eleições Gerais de 2026 em todo o território nacional.\n\n- **Primeiro"
        " Turno:** Realizado no primeiro domingo de outubro de 2026 (04 de"
        " outubro de 2026).\n- **Segundo Turno:** Realizado no último domingo de"
        " outubro de 2026 (25 de outubro de 2026), apenas para os cargos de"
        " Presidente e Governador em unidades federativas onde nenhum candidato"
        " atingir a maioria absoluta dos votos válidos.\n- **Fechamento do"
        " Cadastro Eleitoral:** 151 dias antes do pleito. Nenhum requerimento de"
        " alistamento, transferência ou regularização do título de eleitor"
        " será aceito após esta data.\n- **Registro de Candidaturas:** O prazo"
        " limite para a transmissão dos dados dos candidatos pelos partidos e"
        " federações ao TSE encerra-se às 19h do dia 15 de agosto de 2026.\n-"
        " **Diplomação:** Os candidatos eleitos devem ser diplomados pela"
        " Justiça Eleitoral até o dia 18 de dezembro de 2026.\n"
    ),
    "cargos_e_votação.md": (
        "# Eleições Gerais 2026 · Guia Eleitoral · Seção 2 · Cargos e Ordem de"
        " Votação\n\n## 2.1 Ordem de votação na urna\n\nNas Eleições de 2026, a"
        " ordem de votação no teclado da Urna Eletrônica segue rigorosamente a"
        " sequência determinada pelo TSE:\n\n1. **Deputado Federal** (4"
        " dígitos)\n2. **Deputado Estadual ou Distrital** (5 dígitos)\n3."
        " **Senador — 1ª Vaga** (3 dígitos)\n4. **Senador — 2ª Vaga** (3"
        " dígitos)\n5. **Governador** (2 dígitos)\n6. **Presidente da"
        " República** (2 dígitos)\n\n## 2.2 Renovação do Senado\n\nEm 2026,"
        " haverá a renovação de **dois terços (2/3)** das cadeiras do Senado"
        " Federal. Cada eleitor deverá votar em dois candidatos diferentes para"
        " o Senado. Se o eleitor digitar o mesmo número de candidato para as"
        " duas vagas, o segundo voto será considerado nulo pela urna.\n"
    ),
    "urna_eletronica_specs.md": (
        "# Eleições Gerais 2026 · Manutenção Técnica · Seção 3 · Especificações"
        " da Urna (Modelo UE2026)\n\nO modelo UE2026 traz atualizações na"
        " arquitetura de hardware e segurança criptográfica para o pleito de"
        " 2026.\n\n- **Processador:** Módulo de alta velocidade com suporte a"
        " HSM (Hardware Security Module) integrado.\n- **Alimentação:** Bateria"
        " interna de Lítio-Ferro-Fosfato (LiFePO4) com autonomia mínima de 12"
        " horas de operação contínua sem energia externa.\n- **Mídia de"
        " Carga:** Cartão de memória interno encriptado com chave assimétrica"
        " RSA-4096 e algoritmo SHA-512.\n- **Impressora do BU (Boletim de"
        " Urna):** Módulo térmico acoplado que imprime 5 vias obrigatórias do"
        " Boletim de Urna ao encerramento da votação.\n- **Acessibilidade:**"
        " Teclado em Braille, sintetizador de voz aprimorado para deficientes"
        " visuais e tela auxiliar com intérprete virtual de Libras.\n- **Rede"
        " e Conectividade:** A urna opera sem nenhuma conexão física ou de"
        " rede (offline) durante todo o período de votação. A transmissão de"
        " dados é efetuada apenas após o término do pleito via rede privativa"
        " da Justiça Eleitoral.\n"
    ),
    "procedimentos_secao.md": (
        "# Eleições Gerais 2026 · Manual do Mesário · Seção 4 · Procedimentos de"
        " Seção e Votação\n\nEste documento define o fluxo operacional das"
        " seções eleitorais durante o dia de votação. Cada procedimento possui"
        " um código no registro operacional (Seção 9).\n\n## 4.1 Abertura da"
        " Seção e Emissão da Zerésima (Código PROC-AB1)\n\nÀs 07:00 (horário de"
        " Brasília), os mesários e o presidente de mesa realizam a conferência"
        " dos lacres da urna UE2026. Em seguida, a urna é ligada e emitida a"
        " **Zerésima**, documento que comprova que não há nenhum voto"
        " registrado na máquina para nenhum candidato. O presidente da mesa, os"
        " mesários e os fiscais de partidos presentes assinam o relatório da"
        " Zerésima.\n\n## 4.2 Habilitação do Eleitor e Biometria (Código"
        " PROC-BIO)\n\nO eleitor apresenta documento oficial com foto. O"
        " mesário insere o CPF ou número do título no terminal do mesário. O"
        " eleitor posiciona o polegar ou indicador no leitor biométrico. A urna"
        " permite até **4 tentativas** de leitura biométrica. Se todas"
        " falharem, o mesário pode liberar a votação mediante confirmação do"
        " ano de nascimento do eleitor e digitação de senha de autorização do"
        " presidente de mesa, registrando a ocorrência na ata da seção.\n\n##"
        " 4.3 Encerramento e Emissão de Boletins de Urna (Código PROC-ENC)\n\nÀs"
        " 17:00 (horário de Brasília), o presidente de mesa declara encerrada a"
        " votação, desde que não haja eleitores na fila (caso haja, são"
        " distribuídas senhas). É inserida a Mídia de Memória de Resultado"
        " (MRR) para finalizar o pleito e são impressas as **5 vias"
        " obrigatórias** do Boletim de Urna (BU). Uma via é afixada na porta da"
        " seção para consulta pública imediata e as demais seguem para a junta"
        " eleitoral junto com a mídia de gravação.\n\n## 4.4 Contingência e"
        " Troca de Urna (Código PROC-CONT)\n\nEm caso de falha irrecuperável do"
        " hardware da urna durante a votação, o suporte técnico é acionado. Se"
        " a reinicialização falhar, a urna é substituída por uma **Urna de"
        " Contingência**. O cartão de memória da urna com defeito é"
        " transferido para a nova urna, garantindo que nenhum voto gravado seja"
        " perdido.\n"
    ),
    "financiamento_e_prestacao.md": (
        "# Eleições Gerais 2026 · Manual de Prestação de Contas · Seção 5 ·"
        " Regras de Financiamento\n\n## 5.1 Fundo Especial de Financiamento de"
        " Campanha (FEFC)\n\nO Fundo Eleitoral é distribuído entre os partidos"
        " com base na representação parlamentar na Câmara dos Deputados e no"
        " Senado Federal. Os recursos devem ser mantidos em conta bancária"
        " específica e auditável, aberta exclusivamente para a campanha de"
        " 2026.\n\n## 5.2 Doações de Pessoas Físicas\n\nPessoas físicas podem"
        " doar para candidatos até o limite de **10% do rendimento bruto**"
        " declarado à Receita Federal no ano-calendário anterior (2025). Doações"
        " acima deste limite sujeitam o doador à multa de até 100% do valor"
        " excedente. Doações efetuadas via Pix só serão aceitas se a chave for"
        " o CPF do doador.\n\n## 5.3 Vedações Absolutas\n\nÉ estritamente"
        " proibido o financiamento de campanha por:\n- Pessoas jurídicas"
        " (empresas e corporações).\n- Entidades ou governos estrangeiros.\n-"
        " Concessionárias de serviços públicos e entidades sindicais.\n"
    ),
    "propaganda_e_midia.md": (
        "# Eleições Gerais 2026 · Legislação Eleitoral · Seção 6 · Propaganda e"
        " Redes Sociais\n\n## 6.1 Horário Gratuito de Rádio e TV\n\nO horário"
        " eleitoral gratuito começa 35 dias antes do antevéspera do primeiro"
        " turno. A divisão do tempo destina 10% de forma igualitária entre todos"
        " os partidos e 90% proporcionalmente ao número de representantes na"
        " Câmara dos Deputados.\n\n## 6.2 Regras para Inteligência Artificial e"
        " Deepfakes\n\nO uso de Inteligência Artificial em peças de propaganda"
        " exige **rotulagem explícita e clara** informando que o conteúdo foi"
        " gerado ou alterado sinteticamente. É terminantemente proibido o uso"
        " de *deepfakes* para simular a fala ou imagem de candidatos criando"
        " declarações falsas. A violação acarreta o cancelamento do registro de"
        " candidatura e inelegibilidade do responsável.\n\n## 6.3"
        " Impulsionamento de Conteúdo\n\nO impulsionamento de publicações na"
        " internet só é permitido se realizado diretamente nas plataformas de"
        " redes sociais e contratado exclusivamente por candidatos, partidos ou"
        " coligações, sendo obrigatório a identificação com a marca \"Propaganda"
        " Eleitoral\". É proibido o uso de disparos em massa via aplicativos de"
        " mensagaria sem consentimento prévio do destinatário.\n"
    ),
    "codigos_procedimento.md": (
        "# Eleições Gerais 2026 · Guia Operacional · Seção 9 · Tabela de"
        " Códigos de Procedimento\n\nCódigos de referência padronizados para"
        " comunicação oficial entre seções eleitorais, cartórios e juntas do"
        " TRE.\n\n| Código | Procedimento | Responsável pela Execução |\n|---|"
        "---|---|\n| PROC-AB1 | Emissão de Zerésima e Abertura | Presidente da"
        " Mesa |\n| PROC-BIO | Habilitação Biométrica do Eleitor | 1º Mesário"
        " |\n| PROC-ENC | Encerramento e Impressão do BU | Presidente da Mesa"
        " |\n| PROC-CONT | Substituição por Urna de Contingência | Técnico do"
        " TRE |\n| PROC-APUR | Transmissão de Dados do Resultado | Operador de"
        " Transmissão |\n| PROC-AUD | Auditoria de Urnas por Amostragem | Comissão"
        " de Auditoria |\n\nQualquer atualização nas rotinas operacionais gera"
        " a revisão do sufixo do código (exemplo: PROC-AB1 atualizado passa a"
        " ser PROC-AB2).\n"
    ),
    "glossario_eleitoral.md": (
        "# Eleições Gerais 2026 · Manual de Comunicação · Seção 7 · Glossário de"
        " Termos Oficiais\n\nTermos padronizados utilizados pela Justiça"
        " Eleitoral durante a transmissão e apuração de dados:\n\n-"
        " **\"Zerésima.\"** Relatório impresso pela Urna Eletrônica antes do"
        " início da votação demonstrando que o total de votos acumulados é zero.\n-"
        " **\"Boletim de Urna (BU).\"** Documento oficial impresso ao final da"
        " votação que registra o total de votos recebidos por candidato,"
        " brancos e nulos em uma seção.\n- **\"Quociente Eleitoral.\"** Resultado"
        " da divisão do número de votos válidos pelo número de vagas a preencher"
        " na eleição proporcional (Deputados).\n- **\"Mídia de Memória de\""
        " Resultado (MRR).\"** Dispositivo de armazenamento seguro usado para"
        " transportar os dados da urna para o ponto de transmissão.\n-"
        " **\"Voto de Legenda.\"** Voto digitado apenas com o número do partido"
        " na eleição proporcional, atribuído à sigla e não a um candidato"
        " específico.\n"
    ),
    "fiscalizacao_e_auditoria.md": (
        "# Eleições Gerais 2026 · Segurança da Votação · Seção 8 · Auditoria e"
        " Fiscalização\n\n## 8.1 Teste de Integridade (Votação Paralela)\n\nNo"
        " mesmo dia da eleição, o TRE sorteia seções eleitorais em todo o"
        " Estado para a realização do **Teste de Integridade em Tempo Real**."
        " As urnas sorteadas são trazidas à sede do TRE e submetidas a uma"
        " votação simulada simultânea à votação oficial, com digitação de"
        " cédulas físicas preenchidas por representantes de entidades da"
        " sociedade civil, sendo tudo gravado em áudio e vídeo.\n\n## 8.2"
        " Auditoria do Código-Fonte\n\nOs sistemas e códigos-fonte da Urna"
        " Eletrônica são abertos para inspeção por partidos políticos, Forças"
        " Armadas, OAB, Polícia Federal e universidades até 12 meses antes do"
        " pleito na sala de apuração do Tribunal Superior Eleitoral.\n"
    ),
}

# Saving files in markdown
for name, text in CORPUS.items():
    file_path = os.path.join(CORPUS_DIR, name)
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(text)

#Chunking

In [42]:
# Installing langchain
!pip install -q langchain-text-splitters

In [43]:
import os
from langchain_text_splitters import (
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
)

def chunking(folder=CORPUS_DIR, chunk_size=512, overlap=50):
    headers_to_split_on = [
        ("#", "title"),
        ("##", "subtitle"),
        ("###", "section"),
    ]

    markdown_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=headers_to_split_on, strip_headers=False
    )

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=overlap
    )

    rows = []

    for filename in sorted(os.listdir(folder)):
        if not filename.endswith(".md"):
            continue

        filepath = os.path.join(folder, filename)
        with open(filepath, encoding="utf-8") as f:
            text = f.read()

        docs_md = markdown_splitter.split_text(text)

        chunks = text_splitter.split_documents(docs_md)
        for i, c in enumerate(chunks):
            section_name = (
                c.metadata.get("section")
                or c.metadata.get("subtitle")
                or c.metadata.get("title")
                or "general"
            )

            rows.append({
                "id": f"{filename}#{i}",
                "text": c.page_content,
                "doc": filename,
                "chunk": i,
                "section": section_name,
            })

    return rows

In [44]:
def sources_used(hits):
    """The list of sources comes from the retrieval process and assembles the format [doc · section]."""
    seen, out = set(), []
    for h in hits:
        key = f"{h['doc']} · {h['section']}"
        if key not in seen:
            seen.add(key)
            out.append(key)
    return out

#Inserting Data

In [45]:
rows = chunking()
print(f"Number of chunks: {len(rows)}")

collection_name = "eleicoes_2026_col"
col = build_index(rows, collection_name, rebuild=True)
print(f"Vector index successfully built! Total number of items in the collection: {col.count()}")

Number of chunks: 26
Vector index successfully built! Total number of items in the collection: 26


# Asking RAG

In [46]:
def ask_rag(question, k=4):
    # 1. Retrieve the most relevant chunks from ChromaDB
    hits = retrieve(col, question, k=k)

    # 2. Construct the context using the retrieved texts and their respective sources.
    context_parts = []
    for h in hits:
        context_parts.append(f"Documento: {h['doc']} (Seção: {h['section']})\nConteúdo: {h['text']}")
    context = "\n\n---\n\n".join(context_parts)

    # 3. Construct the system and user prompts for the LLM.
    system_prompt = (
        "Você é um assistente especialista nas Eleições Gerais de 2026. "
        "Responda à pergunta do usuário utilizando estritamente o contexto fornecido abaixo. "
        "Se a resposta não puder ser encontrada no contexto, informe que não possui informações suficientes."
    )

    user_prompt = f"""Contexto:\n{context}\n\nPergunta: {question}\nResposta:"""

    # 4. Call Qwen 2.5 model via the Ollama API (OpenAI client)
    response = client.chat.completions.create(
        model=GEN_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.1
    )

    answer = response.choices[0].message.content
    sources = sources_used(hits)

    return answer, sources

In [47]:
# Realize sua consulta aqui
question = "Qual o dia do primeiro turno?"

answer, sources = ask_rag(question, k=3)

print(f"📌 Pergunta: {question}\n")
print(f"🤖 Resposta:\n{answer}\n")
print("📚 Fontes Consultadas:")
for source in sources:
    print(f"  - {source}")

📌 Pergunta: Qual o dia do primeiro turno?

🤖 Resposta:
O primeiro turno das Eleições Gerais de 2026 ocorreu no primeiro domingo de outubro de 2026, que foi no dia 04 de outubro de 2026.

📚 Fontes Consultadas:
  - datas_e_prazos.md · Eleições Gerais 2026 · Guia de Calendário · Seção 1 · Datas e Prazos Oficiais
  - urna_eletronica_specs.md · Eleições Gerais 2026 · Manutenção Técnica · Seção 3 · Especificações da Urna (Modelo UE2026)


#Questions

In [48]:
import json, os, re, unicodedata
os.makedirs("results", exist_ok=True)

def norm(s):
    s = unicodedata.normalize("NFKD", str(s).lower())
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", s.replace("*", ""))

def Q(id, tier, question, must_contain, gold_docs, expect_refusal=False, history=None):
    return {"id": id, "tier": tier, "question": question, "must_contain": must_contain,
            "gold_docs": gold_docs, "expect_refusal": expect_refusal, "history": history or []}

DATAS, FIN, PROC, COD = "datas_e_prazos.md", "financiamento_e_prestacao.md", "procedimentos_secao.md", "codigos_procedimento.md"
URNA, PROP = "urna_eletronica_specs.md", "propaganda_e_midia.md"
CARGOS = next(k for k in CORPUS if k.startswith("cargos_e_vota"))

QUESTIONS = [
    # ---------------- direct (4) ----------------
    Q("q1", "direct", "Em que dia será realizado o primeiro turno das Eleições Gerais de 2026?",
      ["04 de outubro", "2026"], [DATAS]),
    Q("q2", "direct", "Qual é a ordem de votação no teclado da Urna Eletrônica em 2026?",
      ["Deputado Federal", "Presidente"], [CARGOS]),
    Q("q5", "direct", "Qual é a autonomia mínima da bateria da urna UE2026?",
      ["12 horas"], [URNA]),
    Q("q6", "direct", "Até que dia os candidatos eleitos devem ser diplomados?",
      ["18 de dezembro"], [DATAS]),

    # ---------------- multi (4): vários trechos / documentos ----------------
    Q("q3", "multi", "Quais são os limites de doação de pessoas físicas e quais as restrições sobre doações via Pix?",
      ["10%", "rendimento bruto", "cpf"], [FIN]),
    Q("q7", "multi", "Qual o código do procedimento de substituição por urna de contingência e quem o executa?",
      ["PROC-CONT", "Técnico do TRE"], [PROC, COD]),
    Q("q8", "multi", "Quem executa a habilitação biométrica do eleitor e quantas tentativas de leitura a urna permite?",
      ["1º mesário|primeiro mesário", "4 tentativas|quatro tentativas"], [PROC, COD]),
    Q("q9", "multi", "A que horas a votação é encerrada, quantas vias do Boletim de Urna são impressas e quem é responsável por esse procedimento?",
      ["17:00|17h|17 horas", "5 vias|cinco vias", "presidente d"], [PROC, COD]),

    # ---------------- paraphrase (3): lacuna de vocabulário ----------------
    Q("q10", "paraphrase", "Se a máquina de votar pifar no meio da eleição, o que acontece com os votos que já tinham sido registrados?",
      ["urna de contingência", "cartão de memória"], [PROC]),
    Q("q11", "paraphrase", "Sindicatos e governos de outros países podem bancar campanhas eleitorais?",
      ["sindica", "estrangeir"], [FIN]),
    Q("q12", "paraphrase", "Posso usar um vídeo feito por computador imitando a voz de um candidato dizendo algo que ele nunca falou?",
      ["proibid", "inelegibilidade|cancelamento"], [PROP]),

    # ---------------- specific (3): caso concreto -> regra geral (Step-Back) ----------------
    Q("q13", "specific", "Se um eleitor digitar 123 nas duas vagas de senador, o que acontece com o segundo voto?",
      ["nulo"], [CARGOS]),
    Q("q14", "specific", "Um doador pessoa física pagou a doação via Pix usando a chave CNPJ da empresa dele. A doação será aceita?",
      ["cpf"], [FIN]),
    Q("q15", "specific", "Um fiscal de partido chegou à seção às 07:00 e quer saber qual documento comprova que a urna não tem nenhum voto antes de começar. Qual é?",
      ["zerésima"], [PROC]),

    # ---------------- followup (3): só fazem sentido com o histórico (RAG with memory) ----------------
    Q("q16", "followup", "E para quais cargos ele acontece?",
      ["Presidente", "Governador"], [DATAS],
      history=[{"role": "user", "content": "Quando será o segundo turno das Eleições Gerais de 2026?"},
               {"role": "assistant", "content": "O segundo turno será realizado em 25 de outubro de 2026."}]),
    Q("q17", "followup", "E o que acontece com quem passar desse valor?",
      ["multa", "100%"], [FIN],
      history=[{"role": "user", "content": "Qual o limite de doação de pessoas físicas para candidatos?"},
               {"role": "assistant", "content": "Até 10% do rendimento bruto declarado à Receita Federal no ano anterior."}]),
    Q("q18", "followup", "E se todas elas falharem, como o mesário libera a votação?",
      ["ano de nascimento", "senha"], [PROC],
      history=[{"role": "user", "content": "Quantas tentativas de leitura biométrica a urna permite?"},
               {"role": "assistant", "content": "A urna permite até 4 tentativas de leitura biométrica."}]),

    # ---------------- none (3): a resposta não está no corpus ----------------
    Q("q4", "none", "Qual é a previsão do tempo para o dia da eleição em São Paulo?", [], [], expect_refusal=True),
    Q("q19", "none", "Qual é o valor total em reais do Fundo Eleitoral (FEFC) distribuído em 2026?", [], [], expect_refusal=True),
    Q("q20", "none", "Quais são os nomes dos candidatos a Presidente da República em 2026?", [], [], expect_refusal=True),
]
QUESTIONS.sort(key=lambda q: int(q["id"][1:]))

with open("questions.json", "w", encoding="utf-8") as f:
    json.dump(QUESTIONS, f, ensure_ascii=False, indent=2)

# ---- checagens do gabarito ----
assert len(QUESTIONS) == 20 and len({q["id"] for q in QUESTIONS}) == 20
from collections import Counter as _C
print("perguntas por tier:", dict(_C(q["tier"] for q in QUESTIONS)))

problems = 0
for q in QUESTIONS:
    for d in q["gold_docs"]:
        assert d in CORPUS, f"{q['id']}: gold_doc inexistente: {d}"
    gold_text = norm(" ".join(CORPUS[d] for d in q["gold_docs"]))
    for kw in q["must_contain"]:
        if not any(norm(alt) in gold_text for alt in kw.split("|")):
            problems += 1
            print(f"⚠️  {q['id']}: '{kw}' não aparece nos gold_docs -> gabarito não verificável no corpus")
print("gabarito verificável no corpus ✅" if problems == 0 else f"{problems} problema(s) no gabarito")

perguntas por tier: {'direct': 4, 'multi': 4, 'none': 3, 'paraphrase': 3, 'specific': 3, 'followup': 3}
gabarito verificável no corpus ✅


#Techniques

### Configs

In [49]:
import time, json, re
import numpy as np
import pandas as pd
from collections import defaultdict

# ============== PARAMS CONFIG ==============
K            = 4      # chunks entregues ao gerador
N_RUNS       = 1      # execuções por variante  -> results/<variante>_r1..r3.csv
SEED_BASE    = 42     # seed = SEED_BASE + 1000*run + índice da pergunta (reprodutível e diferente a cada run)
GEN_TEMP     = 0.3    # temperatura da resposta final
REWRITE_TEMP = 0.3    # temperatura das chamadas de reescrita de query
N_VARIANTS   = 5      # reformulações extras no Multi-Query
RRF_C        = 60     # constante do Reciprocal Rank Fusion
MEMORY_DISTRACTOR = True   # perguntas sem histórico recebem um histórico irrelevante (testa contaminação por tema)
FORCE        = True  # True = ignora CSVs existentes e refaz tudo
# ========================================================================================

### General

In [50]:
class Trace:
    def __init__(self):
        self.llm_calls = 0
        self.embed_calls = 0
        self.queries = []

In [51]:
def llm(messages, tr, temperature, seed):
    tr.llm_calls += 1
    r = client.chat.completions.create(model=GEN_MODEL, messages=messages,
                                       temperature=temperature, seed=seed)
    return r.choices[0].message.content.strip()

def search(col, query, tr, k=K):
    tr.embed_calls += 1
    tr.queries.append(query)
    res = col.query(query_embeddings=embed([query]), n_results=k,
                    include=["documents", "metadatas", "distances"])
    return [{"id": i, "text": t, "doc": m["doc"], "section": m["section"], "distance": d}
            for i, t, m, d in zip(res["ids"][0], res["documents"][0], res["metadatas"][0], res["distances"][0])]

def rrf(rankings, c=RRF_C):
    """Reciprocal Rank Fusion"""
    fused = defaultdict(float)
    for ranking in rankings:
        for rank, cid in enumerate(ranking, 1):
            fused[cid] += 1.0 / (c + rank)
    return sorted(fused.items(), key=lambda x: -x[1])

def dedupe_topk(hits, k=K):
    best = {}
    for h in hits:
        if h["id"] not in best or h["distance"] < best[h["id"]]["distance"]:
            best[h["id"]] = h
    return sorted(best.values(), key=lambda h: h["distance"])[:k]

def clean_lines(text, original, n):
    out = []
    for line in text.splitlines():
        line = re.sub(r"^\s*(?:[-*•]|\d+[.)])\s*", "", line).strip().strip('"')
        if line and norm(line) != norm(original) and len(line) < 300 and line not in out:
            out.append(line)
    return out[:n]

In [52]:
SYSTEM_ANSWER = ("Você é um assistente especialista nas Eleições Gerais de 2026. "
                 "Responda usando APENAS o contexto abaixo, de forma direta e em português. "
                 "Se a resposta não estiver no contexto, diga que não possui informações suficientes.")

def format_context(hits):
    return "\n\n".join(f"[{h['doc']} · {h['section']}]\n{h['text']}" for h in hits)

def generate_answer(question, hits, tr, seed):
    messages = [{"role": "system", "content": SYSTEM_ANSWER},
                {"role": "user", "content": f"Contexto:\n{format_context(hits)}\n\nPergunta: {question}"}]
    return llm(messages, tr, GEN_TEMP, seed)

def r_baseline(q, col, tr, seed):
    return q["question"], search(col, q["question"], tr)

### Stepback

In [53]:
STEPBACK_SYSTEM = (
    "Você reformula perguntas para busca em documentos. Dada uma pergunta específica, escreva UMA pergunta "
    "mais geral (step-back) sobre a regra, o princípio ou o assunto mais amplo por trás dela. "
    "Responda apenas com a pergunta geral.\n\n"
    "Exemplo 1\nPergunta: Meu carro foi multado às 23h em Recife por excesso de 12 km/h. Posso recorrer?\n"
    "Pergunta geral: Quais são as regras e prazos para recorrer de multas de trânsito?\n\n"
    "Exemplo 2\nPergunta: O aluno João faltou a 8 aulas de Cálculo. Ele reprova?\n"
    "Pergunta geral: Qual é o limite de faltas permitido para aprovação em uma disciplina?"
)

def r_stepback(q, col, tr, seed):
    out = llm([{"role": "system", "content": STEPBACK_SYSTEM},
               {"role": "user", "content": f"Pergunta: {q['question']}\nPergunta geral:"}],
              tr, REWRITE_TEMP, seed)
    lines = [l.strip() for l in out.splitlines() if l.strip()]
    step_q = lines[0] if lines else q["question"]
    hits = search(col, q["question"], tr) + search(col, step_q, tr)
    return q["question"], dedupe_topk(hits, K)

### MultiQuery / Fusion

In [54]:
def r_multiquery(q, col, tr, seed):
    system = (f"Gere {N_VARIANTS} reformulações diferentes da pergunta do usuário para busca em documentos, "
              "usando sinônimos e outros termos (formais e coloquiais). "
              "Uma por linha, sem numeração e sem explicações.")
    out = llm([{"role": "system", "content": system}, {"role": "user", "content": q["question"]}],
              tr, REWRITE_TEMP, seed)
    queries = [q["question"]] + clean_lines(out, q["question"], N_VARIANTS)
    rankings, pool = [], {}
    for qq in queries:
        hs = search(col, qq, tr)
        rankings.append([h["id"] for h in hs])
        for h in hs:
            if h["id"] not in pool or h["distance"] < pool[h["id"]]["distance"]:
                pool[h["id"]] = h
    hits = []
    for cid, score in rrf(rankings)[:K]:
        h = dict(pool[cid]); h["rrf"] = round(score, 5)
        hits.append(h)
    return q["question"], hits

### Memory

In [55]:
DISTRACTOR = [{"role": "user", "content": "O que é o Boletim de Urna?"},
              {"role": "assistant", "content": "É o documento impresso ao final da votação que registra os votos "
                                                "recebidos por candidato, brancos e nulos em uma seção."}]

MEMORY_SYSTEM = ("Dado o histórico da conversa e uma pergunta de acompanhamento, reescreva a pergunta de "
                 "acompanhamento para que seja independente e completa, em português. Se ela já for "
                 "independente, repita-a sem alterações. Responda apenas com a pergunta reescrita.")

def r_memory(q, col, tr, seed):
    history = q.get("history") or (DISTRACTOR if MEMORY_DISTRACTOR else [])
    if not history:
        return q["question"], search(col, q["question"], tr)
    hist = "\n".join(f"{'Usuário' if t['role'] == 'user' else 'Assistente'}: {t['content']}" for t in history[-4:])
    out = llm([{"role": "system", "content": MEMORY_SYSTEM},
               {"role": "user", "content": f"Histórico:\n{hist}\n\nPergunta de acompanhamento: {q['question']}\n\nPergunta independente:"}],
              tr, REWRITE_TEMP, seed)
    lines = [l.strip() for l in out.splitlines() if l.strip()]
    standalone = lines[0] if lines else q["question"]
    return standalone, search(col, standalone, tr)


VARIANTS = {"baseline": r_baseline, "stepback": r_stepback, "multiquery": r_multiquery, "memory": r_memory}

# Evaluate

In [56]:
REFUSAL_RE = re.compile(
    r"nao (possui|possuo|tenho|encontr|ha informac|consta|foi possivel|fornec|menciona)"
    r"|informacoes suficientes|nao (esta|estao) (disponivel|disponiveis|mencionad|presente)|nao (e |ha )?mencionad")

def has_kw(text_norm, kw):
    return any(norm(alt).strip() in text_norm for alt in kw.split("|"))

def grade(answer, q):
    """-> (correct, refused). Perguntas respondíveis: todas as must_contain presentes. Perguntas 'none': recusar = correto."""
    a = norm(answer)
    refused = bool(REFUSAL_RE.search(a))
    if q["expect_refusal"]:
        return refused, refused
    return all(has_kw(a, kw) for kw in q["must_contain"]), refused

def run_variant(label, fn, questions, run, col):
    path = f"results/{label}_r{run}.csv"
    if os.path.exists(path) and not FORCE:
        return pd.read_csv(path)
    rows = []
    for qi, q in enumerate(questions):
        tr, seed = Trace(), SEED_BASE + 1000 * run + qi
        t0 = time.time()
        q_used, hits = fn(q, col, tr, seed)
        answer = generate_answer(q_used, hits, tr, seed)
        latency = time.time() - t0

        correct, refused = grade(answer, q)
        docs = {h["doc"] for h in hits}
        recall = (sum(d in docs for d in q["gold_docs"]) / len(q["gold_docs"])) if q["gold_docs"] else np.nan
        ctx = norm(format_context(hits))
        ctx_keys = all(has_kw(ctx, kw) for kw in q["must_contain"]) if q["must_contain"] else np.nan

        rows.append({
            "label": label, "run": run, "id": q["id"], "tier": q["tier"], "question": q["question"],
            "question_used": q_used, "queries": json.dumps(tr.queries, ensure_ascii=False),
            "retrieved": " | ".join(f"{h['id']}({h['distance']:.3f})" for h in hits),
            "gold_docs": ";".join(q["gold_docs"]), "gold_recall": recall, "ctx_has_keys": ctx_keys,
            "answer": answer, "correct": correct, "refused": refused,
            "llm_calls": tr.llm_calls, "embed_calls": tr.embed_calls, "latency_s": round(latency, 2),
        })
    df = pd.DataFrame(rows)
    df.to_csv(path, index=False)
    return df

In [57]:
with open("questions.json", encoding="utf-8") as f:
    questions = json.load(f)

# aquecimento: a 1ª chamada carrega o modelo na GPU e distorceria a latência da 1ª pergunta
llm([{"role": "user", "content": "ok"}], Trace(), 0.0, 0); embed(["ok"])

for label, fn in VARIANTS.items():
    for run in range(1, N_RUNS + 1):
        t0 = time.time()
        df = run_variant(label, fn, questions, run, col)
        print(f"{label:<11} r{run}  acc={df['correct'].mean():.0%}  llm_calls/pergunta={df['llm_calls'].mean():.1f}  ({time.time()-t0:.0f}s)")
    pd.concat([pd.read_csv(f"results/{label}_r{r}.csv") for r in range(1, N_RUNS + 1)]) \
      .to_csv(f"results/results_{label}.csv", index=False)

# configuração do experimento (para o README / reprodutibilidade)
try:
    ollama_version = requests.get(f"{OLLAMA_URL}/api/version", timeout=3).json().get("version")
except Exception:
    ollama_version = None
with open("results/run_config.json", "w", encoding="utf-8") as f:
    json.dump({"gen_model": GEN_MODEL, "emb_model": EMB_MODEL, "ollama_version": ollama_version,
               "k": K, "n_runs": N_RUNS, "seed_base": SEED_BASE, "gen_temp": GEN_TEMP,
               "rewrite_temp": REWRITE_TEMP, "n_variants": N_VARIANTS, "rrf_c": RRF_C,
               "memory_distractor": MEMORY_DISTRACTOR, "n_chunks": len(rows), "n_questions": len(questions),
               "date": time.strftime("%Y-%m-%d %H:%M")}, f, indent=2, ensure_ascii=False)
print("\nOK -> results/")

baseline    r1  acc=70%  llm_calls/pergunta=1.0  (15s)
stepback    r1  acc=75%  llm_calls/pergunta=2.0  (22s)
multiquery  r1  acc=75%  llm_calls/pergunta=2.0  (43s)
memory      r1  acc=75%  llm_calls/pergunta=2.0  (24s)

OK -> results/
